# Fitting & Training Pipeline

This notebook takes the dataset and uses it to train 3 models, each tuned with cross-validated
hyperparameter search, explicit overfitting controls, and an explicit final-training step:

1. Random Forest
2. SVM
3. XGBoost

## Responsibilities of this notebook
- Load the pre-split train/test data, the `StratifiedKFold` splitter, and the preprocessing
  template produced by an earlier notebook.
- For each model: run randomized hyperparameter search with k-fold CV (regularization-focused
  search spaces, plus early stopping for XGBoost) to find the best hyperparameters.
- Explicitly (re)train each model's final pipeline on the **full** training set using those
  winning hyperparameters.
- Evaluate each final model once on the held-out test set.
- Persist the fitted pipelines, full CV results, and test-set metrics under `../Models/` and
  `../Results/` so they can be reloaded and analyzed later without rerunning training.

## 1. Imports and configuration

In [ ]:
import json, os, joblib
import numpy as np
import pandas as pd
from scipy.stats import loguniform

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
DUMPS_DIR = "../Dumps/"
RESULTS_DIR = "../Results/"
MODELS_DIR = "../Models/"
DATAFRAMEs_DIR = "../DataFrames/"
CV_RESULTS_DIR = os.path.join(RESULTS_DIR, "cv_results")
METRICS_DIR = os.path.join(RESULTS_DIR, "metrics")

TARGET = "Churn"
RANDOM_STATE = 42

N_ITER_SEARCH = 40          # RandomizedSearchCV candidates per model (SVM uses fewer, see below)
EARLY_STOPPING_ROUNDS = 30  # XGBoost only
ES_VALIDATION_SIZE = 0.15   # fraction of X_train carved out to monitor early stopping

for d in (DUMPS_DIR, RESULTS_DIR, MODELS_DIR, CV_RESULTS_DIR, METRICS_DIR, DATAFRAMEs_DIR):
    os.makedirs(d, exist_ok=True)

## 2. Load all required data

In [ ]:
X_train = pd.read_parquet(os.path.join(DATAFRAMEs_DIR, 'X_train.parquet'))
y_train = pd.read_parquet(os.path.join(DATAFRAMEs_DIR, 'y_train.parquet'))

X_test = pd.read_parquet(os.path.join(DATAFRAMEs_DIR, 'X_test.parquet'))
y_test = pd.read_parquet(os.path.join(DATAFRAMEs_DIR, 'y_test.parquet'))


def _to_1d_target(y):
    """y_*.parquet loads as a 1-column DataFrame; sklearn wants a 1-D array/Series.
    Maps string labels ('No'/'Yes') to binary integers (0/1) for metric evaluation,
    XGBoost compatibility, and class weighting.
    """
    if isinstance(y, pd.DataFrame):
        y = y[TARGET] if TARGET in y.columns else y.iloc[:, 0]
    y = y.squeeze()
    if not pd.api.types.is_numeric_dtype(y):
        y = y.map({"No": 0, "Yes": 1, "0": 0, "1": 1, False: 0, True: 1}).astype(int)
    return y


y_train = _to_1d_target(y_train)
y_test = _to_1d_target(y_test)

print(f"X_train Length: {len(X_train)}, y_train Length: {len(y_train)}")
print(f"X_test Length: {len(X_test)}, y_test Length: {len(y_test)}")
print("\nTrain class balance:")
print(y_train.value_counts(normalize=True))


In [ ]:
# Preprocessor definition (constructed natively to avoid scikit-learn version mismatch when unpickling)
def build_preprocessor():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("yeo_johnson", PowerTransformer(method="yeo-johnson")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="if_binary", handle_unknown="ignore")),
    ])
    numeric_cols = [
        "tenure", "MonthlyCharges", "TotalCharges", "ServiceCount",
        "StreamingCount", "SecuritySupportCount", "AverageMonthlySpend"
    ]
    binary_numeric_cols = [
        "SeniorCitizen", "IsMonthToMonth", "IsElectronicCheck",
        "HasFamily", "IsNewCustomer", "NewCustomer_HighCharge", "IsOldCustomer"
    ]
    categorical_cols = [
        "gender", "Partner", "Dependents", "PhoneService", "PaperlessBilling",
        "MultipleLines", "InternetService", "OnlineSecurity", "OnlineBackup",
        "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
        "Contract", "PaymentMethod", "MonthlyChargesGroup", "TotalChargesGroup", "tenureGroup"
    ]
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_cols),
        ("binary_numeric", "passthrough", binary_numeric_cols),
        ("categorical", categorical_pipeline, categorical_cols),
    ])

# Load or instantiate preprocessor template
try:
    preprocessor = joblib.load(os.path.join(DUMPS_DIR, 'preprocessor_template.pkl'))
    # Test if get_params() works without version mismatch errors
    _ = preprocessor.get_params()
    print("Loaded preprocessor from template pickle.")
except Exception as e:
    print(f"Rebuilding preprocessor natively to prevent version mismatch ({e})...")
    preprocessor = build_preprocessor()

# Stratified K-Fold splitter
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("skf:", skf)
preprocessor

## 3. Tuning / final-training / evaluation / persistence helpers

Three separate helpers, one per responsibility, so the notebook makes it obvious *when* and *on
what data* each thing happens:

- **`search_hyperparameters`** — runs `RandomizedSearchCV` (on the shared `skf` folds) to find
  the best hyperparameters. `refit="roc_auc"` is required by sklearn for `best_params_` /
  `best_score_` to be populated at all, but the pipeline that internal refit produces is
  discarded — we never touch `search.best_estimator_`. The real final-training step happens
  explicitly, next.
- **`train_final_model`** — clones the untouched pipeline template, applies the winning
  hyperparameters, and explicitly calls `.fit(X_train, y_train)` on the **entire** training set.
  This is the actual final-model-training step.
- **`evaluate_and_save`** — scores the already-fitted final pipeline once on the held-out test
  set, then persists the pipeline + metrics to `../Models/` and `../Results/`.

In [ ]:
def search_hyperparameters(name, pipeline, param_distributions, n_iter=N_ITER_SEARCH):
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=n_iter,
        scoring=["roc_auc", "precision", "recall", "f1"],
        refit="roc_auc",   # required for best_params_/best_score_ to be populated; the pipeline
                           # this triggers internally is discarded — train_final_model() below is
                           # the real, explicit final-training step
        cv=skf,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1,
        return_train_score=True,
    )
    search.fit(X_train, y_train)

    pd.DataFrame(search.cv_results_).sort_values("rank_test_roc_auc").to_csv(
        os.path.join(CV_RESULTS_DIR, f"{name}_cv_results.csv"), index=False
    )
    print(f"[{name}] best CV ROC AUC: {search.best_score_:.4f}")
    print(f"[{name}] best params: {search.best_params_}")
    return search


def train_final_model(pipeline_template, best_params):
    """Explicitly (re)train the final model on the FULL training set using the winning
    hyperparameters found by search_hyperparameters()."""
    final_pipeline = clone(pipeline_template).set_params(**best_params)
    final_pipeline.fit(X_train, y_train)
    return final_pipeline


def evaluate_and_save(name, fitted_pipeline, extra_info=None):
    y_pred = fitted_pipeline.predict(X_test)
    y_proba = fitted_pipeline.predict_proba(X_test)[:, 1]

    test_metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    }
    if extra_info:
        test_metrics.update(extra_info)

    joblib.dump(fitted_pipeline, os.path.join(MODELS_DIR, f"{name}_best_pipeline.pkl"))
    with open(os.path.join(METRICS_DIR, f"{name}_test_metrics.json"), "w") as f:
        json.dump(test_metrics, f, indent=2, default=str)

    print(f"\n=== {name.upper()} — held-out test performance ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    print(f"ROC AUC: {test_metrics['roc_auc']:.4f}")

    return fitted_pipeline, test_metrics

# Training the KFold-tuned models

## 1. Random Forest

In [ ]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=1,   # RandomizedSearchCV already parallelizes across candidates/folds (n_jobs=-1 below)
    )),
])

In [ ]:
# Random Forest has no L1/L2 penalty; its regularization knobs are tree complexity
# (max_depth, min_samples_split/leaf, max_features) and how many trees get averaged
# (n_estimators). We search over all of these plus class balancing strategy.
rf_param_distributions = {
    "model__n_estimators": [150, 200, 300, 400, 500],
    "model__max_depth": [4, 6, 8, 10, 12, 16, None],
    "model__min_samples_split": [2, 5, 10, 20],
    "model__min_samples_leaf": [1, 2, 4, 8],
    "model__max_features": ["sqrt", "log2", 0.5, 0.8],
    "model__class_weight": ["balanced", "balanced_subsample"],
}

In [ ]:
rf_search = search_hyperparameters("random_forest", rf_pipeline, rf_param_distributions)

### Train the final Random Forest on the full training set

In [ ]:
rf_final_pipeline = train_final_model(rf_pipeline, rf_search.best_params_)

rf_best_pipeline, rf_test_metrics = evaluate_and_save(
    "random_forest",
    rf_final_pipeline,
    extra_info={"best_cv_roc_auc": rf_search.best_score_, "best_params": rf_search.best_params_},
)

In [ ]:
# importances = pd.Series(
#     rf_best_pipeline.named_steps["model"].feature_importances_,
#     index=rf_best_pipeline.named_steps["preprocessor"].get_feature_names_out(),
# )
# importances.sort_values(ascending=False).head(10)

## 2. SVM

In [ ]:
svc_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVC(
        kernel="rbf",
        probability=True,   # needed for predict_proba -> ROC AUC scoring/evaluation
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )),
])

In [ ]:
# For an RBF SVM, C is the inverse of the L2 regularization strength (small C = more
# regularization, simpler boundary) and gamma controls how tightly the kernel fits individual
# points (small gamma = smoother, less overfit boundary). We search both on a log scale.
svc_param_distributions = {
    "model__C": loguniform(1e-2, 1e2),
    "model__gamma": loguniform(1e-4, 1e0),
}

In [ ]:
# probability=True makes SVC noticeably slower to fit (extra internal CV for Platt scaling),
# so we search fewer candidates than for the other two models.
svc_search = search_hyperparameters("svm", svc_pipeline, svc_param_distributions, n_iter=25)

### Train the final SVM on the full training set

In [ ]:
svc_final_pipeline = train_final_model(svc_pipeline, svc_search.best_params_)

svc_best_pipeline, svc_test_metrics = evaluate_and_save(
    "svm",
    svc_final_pipeline,
    extra_info={"best_cv_roc_auc": svc_search.best_score_, "best_params": svc_search.best_params_},
)

## 3. XGBoost

In [ ]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f"scale_pos_weight (neg/pos): {scale_pos_weight:.3f}")

xgb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        n_jobs=1,
        scale_pos_weight=scale_pos_weight,
    )),
])

In [ ]:
# Overfitting controls for XGBoost:
#  - reg_alpha / reg_lambda: explicit L1 / L2 penalties on leaf weights
#  - subsample / colsample_bytree: stochastic regularization (each tree sees a random subset
#    of rows / columns)
#  - max_depth / min_child_weight / gamma: constrain how complex any single tree can get
#  - early stopping (added further down) caps the number of boosting rounds
xgb_param_distributions = {
    "model__n_estimators": [100, 200, 300, 500],
    "model__max_depth": [3, 4, 5, 6, 8],
    "model__learning_rate": loguniform(1e-2, 3e-1),
    "model__subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 7],
    "model__gamma": [0, 0.1, 0.3, 0.5, 1.0],
    "model__reg_alpha": [0, 0.001, 0.01, 0.1, 1, 5],     # L1
    "model__reg_lambda": [0.1, 1, 2, 5, 10, 20],          # L2
}

In [ ]:
xgb_search = search_hyperparameters("xgboost", xgb_pipeline, xgb_param_distributions)

### Early stopping refinement

`search_hyperparameters` above picked the best structural/regularization hyperparameters, but
every candidate was trained for a fixed `n_estimators`. Here we reuse those winning
hyperparameters and let early stopping choose the actual number of boosting rounds on a held-out
validation slice of the training data, so the final model stops as soon as validation loss stops
improving instead of continuing to fit noise. The test set (`X_test` / `y_test`) is never
touched during this step.

In [ ]:
best_xgb_params = {k.replace("model__", ""): v for k, v in xgb_search.best_params_.items()}
best_xgb_params.pop("n_estimators", None)  # will be superseded by the early-stopped value

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=ES_VALIDATION_SIZE, random_state=RANDOM_STATE, stratify=y_train
)

es_preprocessor = clone(preprocessor).fit(X_tr)
X_tr_enc = es_preprocessor.transform(X_tr)
X_val_enc = es_preprocessor.transform(X_val)

es_model = XGBClassifier(
    **best_xgb_params,
    n_estimators=2000,   # high ceiling; early stopping will cut this short
    random_state=RANDOM_STATE,
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    n_jobs=-1,
)
es_model.fit(X_tr_enc, y_tr, eval_set=[(X_val_enc, y_val)], verbose=False)

best_n_estimators = es_model.best_iteration + 1
print(f"Early stopping selected {best_n_estimators} boosting rounds (cap was 2000).")

### Train the final XGBoost model on the full training set

In [ ]:
final_xgb_params = dict(best_xgb_params)
final_xgb_params["n_estimators"] = best_n_estimators  # picked by early stopping, above

final_xgb_pipeline = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", XGBClassifier(
        **final_xgb_params,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        n_jobs=-1,
    )),
])
final_xgb_pipeline.fit(X_train, y_train)  # explicit final training on the FULL training set

xgb_best_pipeline, xgb_test_metrics = evaluate_and_save(
    "xgboost",
    final_xgb_pipeline,
    extra_info={
        "best_params_from_search": xgb_search.best_params_,
        "best_cv_roc_auc": xgb_search.best_score_,
        "early_stopping_best_n_estimators": best_n_estimators,
    },
)

# Model comparison & results summary

In [ ]:
summary_rows = []
for model_name, metrics in [
    ("Random Forest", rf_test_metrics),
    ("SVM", svc_test_metrics),
    ("XGBoost", xgb_test_metrics),
]:
    summary_rows.append({
        "model": model_name,
        "test_roc_auc": metrics["roc_auc"],
        "test_precision": metrics["precision"],
        "test_recall": metrics["recall"],
        "test_f1": metrics["f1"],
        "test_accuracy": metrics["accuracy"],
        "best_cv_roc_auc": metrics.get("best_cv_roc_auc"),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("test_roc_auc", ascending=False).reset_index(drop=True)
summary_df.to_csv(os.path.join(RESULTS_DIR, "model_comparison_summary.csv"), index=False)

with open(os.path.join(RESULTS_DIR, "model_comparison_summary.json"), "w") as f:
    json.dump(summary_rows, f, indent=2, default=str)

summary_df

## Where everything got saved

```
Models/               fitted final Pipeline objects (preprocessor + final model), one .pkl per model
Results/
  cv_results/          full RandomizedSearchCV cv_results_ for each model, as CSV
                        (every candidate tried, its params, and its per-fold train/val scores)
  metrics/             per-model held-out test metrics + best hyperparameters, as JSON
  model_comparison_summary.csv / .json   one row per model for quick side-by-side comparison
```

Reload a saved final model with `joblib.load("../Models/xgboost_best_pipeline.pkl")` to run
predictions later without retraining.